# 1km HRDPS Conversion to Operational Files

## Grid File Information

All the 1-km HRDPS files are saved on `/results/forcing/atmospheric/GEM1.0/GRIB` from `2020-02-07`. An example path of the file should be `'/results/forcing/atmospheric/GEM1.0/GRIB/20200207/12/001/CMC_hrdps_west_VGRD_TGL_10_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2'`.

Here are the variable names, long names and file names:

### Variables

| File Name | Variable | 变量 | 所在高度  |
| :--- | :--- | :--- | :--- |
| **APCP** | Accumulated Precipitation | **累积降水量** | `SFC_0` (地表) |
| **DLWRF** | Downward Long-Wave Radiation Flux | **向下长波辐射通量** | `SFC_0` (地表) |
| **DSWRF** | Downward Short-Wave Radiation Flux | **向下短波辐射通量** | `SFC_0` (地表) |
| **LHTFL** | Latent Heat Net Flux | **潜热通量** | `SFC_0` (地表) |
| **PRATE** | Precipitation Rate | **降水率** | `SFC_0` (地表) |
| **PRMSL** | Pressure reduced to Mean Sea Level | **海平面气压** | `MSL_0` (平均海平面) |
| **SPFH** | Specific Humidity | **比湿** | `TGL_2` (距地2米) |
| **TCDC** | Total Cloud Cover | **总云量** | `SFC_0` (地表) |
| **TMP** | Temperature | **温度** | `TGL_2` (距地2米) |
| **UGRD** | U-Component of Wind | **U风分量** (纬向风/东西向) | `TGL_10` (距地10米) |
| **VGRD** | V-Component of Wind | **V风分量** (经向风/南北向) | `TGL_10` (距地10米) |

### File Names

1. `20200207/12/001/CMC_hrdps_west_APCP_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

2. `20200207/12/001/CMC_hrdps_west_APCP_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2.5b7b6.idx`

3. `20200207/12/001/CMC_hrdps_west_DLWRF_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

4. `20200207/12/001/CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

5. `20200207/12/001/CMC_hrdps_west_LHTFL_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

6. `20200207/12/001/CMC_hrdps_west_PRATE_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

7. `20200207/12/001/CMC_hrdps_west_PRMSL_MSL_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

8. `20200207/12/001/CMC_hrdps_west_SPFH_TGL_2_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

9. `20200207/12/001/CMC_hrdps_west_TCDC_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

10. `20200207/12/001/CMC_hrdps_west_TMP_TGL_2_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

11. `20200207/12/001/CMC_hrdps_west_UGRD_TGL_10_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

12. `20200207/12/001/CMC_hrdps_west_VGRD_TGL_10_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2`

In [1]:
from pathlib import Path

from eccodes import (
    codes_get,
    codes_grib_new_from_file,
    codes_release,
)


DATA_DIR = Path(
    "/results/forcing/atmospheric/GEM1.0/GRIB/"
    "20200207/12/001"
)


def safe_get(grib_id, key, default="N/A"):
    """安全读取一个 GRIB 元数据字段。"""
    try:
        return codes_get(grib_id, key)
    except Exception:
        return default


def inspect_grib_file(file_path: Path) -> None:
    """显示一个 GRIB2 文件中所有 message 的变量和单位信息。"""
    print(f"\n文件: {file_path.name}")

    message_number = 0

    with file_path.open("rb") as file_obj:
        while True:
            grib_id = codes_grib_new_from_file(file_obj)

            if grib_id is None:
                break

            message_number += 1

            try:
                short_name = safe_get(grib_id, "shortName")
                long_name = safe_get(grib_id, "name")
                units = safe_get(grib_id, "units")

                type_of_level = safe_get(grib_id, "typeOfLevel")
                level = safe_get(grib_id, "level")

                step_type = safe_get(grib_id, "stepType")
                step_range = safe_get(grib_id, "stepRange")

                data_date = safe_get(grib_id, "dataDate")
                data_time = safe_get(grib_id, "dataTime")
                valid_date = safe_get(grib_id, "validityDate")
                valid_time = safe_get(grib_id, "validityTime")

                print(f"  Message       : {message_number}")
                print(f"  shortName     : {short_name}")
                print(f"  long name     : {long_name}")
                print(f"  units         : {units}")
                print(f"  typeOfLevel   : {type_of_level}")
                print(f"  level         : {level}")
                print(f"  stepType      : {step_type}")
                print(f"  stepRange     : {step_range}")
                print(f"  initialization: {data_date} {data_time:04}")
                print(f"  valid time    : {valid_date} {valid_time:04}")

            finally:
                codes_release(grib_id)


def main() -> None:
    grib_files = sorted(DATA_DIR.glob("*.grib2"))

    if not grib_files:
        raise FileNotFoundError(
            f"在目录中没有找到 .grib2 文件：{DATA_DIR}"
        )

    print(f"目录: {DATA_DIR}")
    print(f"找到 {len(grib_files)} 个 GRIB2 文件")

    for file_path in grib_files:
        inspect_grib_file(file_path)


if __name__ == "__main__":
    main()

目录: /results/forcing/atmospheric/GEM1.0/GRIB/20200207/12/001
找到 11 个 GRIB2 文件

文件: CMC_hrdps_west_APCP_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2
  Message       : 1
  shortName     : unknown
  long name     : unknown
  units         : unknown
  typeOfLevel   : surface
  level         : 0
  stepType      : accum
  stepRange     : 0-1
  initialization: 20200207 1200
  valid time    : 20200207 1300

文件: CMC_hrdps_west_DLWRF_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2
  Message       : 1
  shortName     : strd
  long name     : Surface long-wave (thermal) radiation downwards
  units         : J m**-2
  typeOfLevel   : surface
  level         : 0
  stepType      : accum
  stepRange     : 0-1
  initialization: 20200207 1200
  valid time    : 20200207 1300

文件: CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20200207T12Z_P001-00.grib2
  Message       : 1
  shortName     : ssrd
  long name     : Surface short-wave (solar) radiation downwards
  units        

## Example Conversion

In [10]:



from pathlib import Path
import re

import numpy as np
import pandas as pd
import xarray as xr


# ============================================================
# 配置
# ============================================================

GRIB_ROOT = Path("/results/forcing/atmospheric/GEM1.0/GRIB")
TARGET_DATE = pd.Timestamp("2023-03-01")
OUTPUT_FILE = f"HRDPS_1km_y{TARGET_DATE:%Ym%md%d}.nc"

# 输出变量名: (GRIB变量名, 高度)
VAR_MAPPING = {
    "solar":     ("DSWRF", "SFC_0"),
    "precip":    ("APCP",  "SFC_0"),
    "therm_rad": ("DLWRF", "SFC_0"),
    "tair":      ("TMP",   "TGL_2"),
    "qair":      ("SPFH",  "TGL_2"),
    "u_wind":    ("UGRD",  "TGL_10"),
    "v_wind":    ("VGRD",  "TGL_10"),
    "atmpres":   ("PRMSL", "MSL_0"),
}

UNITS = {
    "solar": "W m-2",
    "precip": "kg m-2 s-1",
    "therm_rad": "W m-2",
    "tair": "K",
    "qair": "kg kg-1",
    "u_wind": "m s-1",
    "v_wind": "m s-1",
    "atmpres": "Pa",
    "percentcloud": "percent",
}


def parse_filename(path):
    """从文件名读取初始化时间、预报时效和有效时间。"""

    match = re.search(
        r"_(\d{8})T(\d{2})Z_P(\d{3})-00\.grib2$",
        path.name,
    )

    if match is None:
        return None

    init_time = pd.to_datetime(
        match.group(1) + match.group(2),
        format="%Y%m%d%H",
    )

    lead_hour = int(match.group(3))
    valid_time = init_time + pd.Timedelta(hours=lead_hour)

    return init_time, lead_hour, valid_time


def find_file(grib_name, level_name, valid_time):
    """
    找到指定有效时间的文件。

    如果有多个文件，优先选择预报时效最短的文件，
    例如优先选择P001而不是P012。
    """

    candidates = []

    # 搜索目标日前一天和目标日
    for date in [
        TARGET_DATE - pd.Timedelta(days=1),
        TARGET_DATE,
    ]:
        folder = GRIB_ROOT / date.strftime("%Y%m%d")

        pattern = (
            f"*/*/CMC_hrdps_west_{grib_name}_{level_name}_"
            f"rotated_latlon0.009x0.009_*.grib2"
        )

        for path in folder.glob(pattern):
            info = parse_filename(path)

            if info is None:
                continue

            init_time, lead_hour, file_valid_time = info

            if file_valid_time == valid_time:
                candidates.append(
                    (lead_hour, -init_time.timestamp(), path)
                )

    if not candidates:
        raise FileNotFoundError(
            f"找不到 {grib_name} 在 {valid_time} 的文件"
        )

    candidates.sort()

    lead_hour, _, path = candidates[0]

    return path, lead_hour


def read_grib(path):
    """读取一个GRIB文件。"""

    ds = xr.open_dataset(
        path,
        engine="cfgrib",
        backend_kwargs={"indexpath": ""},
    )

    variable_name = list(ds.data_vars)[0]
    da = ds[variable_name].squeeze(drop=True).load()

    data = da.values.astype(np.float32)

    # 经纬度按照现有HRDPS文件的格式：
    # 1. 使用float64
    # 2. 经度使用0–360度范围
    lat = ds["latitude"].values.astype(np.float64)
    lon = np.mod(
        ds["longitude"].values.astype(np.float64),
        360.0,
    )

    ds.close()

    return data, lat, lon


# ============================================================
# 读取24小时数据
# ============================================================

times = pd.date_range(
    TARGET_DATE,
    periods=24,
    freq="h",
)

all_data = {}
nav_lat = None
nav_lon = None

for target_var, (grib_name, level_name) in VAR_MAPPING.items():

    print(f"\n处理 {target_var}")

    hourly_data = []

    for valid_time in times:

        path, lead_hour = find_file(
            grib_name,
            level_name,
            valid_time,
        )

        data, lat, lon = read_grib(path)

        # 累计时间长度
        accumulation_seconds = lead_hour * 3600.0

        if accumulation_seconds <= 0:
            raise ValueError(
                f"无效预报时效：{path.name}"
            )

        # J m-2 -> W m-2
        if target_var in ["solar", "therm_rad"]:
            data = data / accumulation_seconds

        # 累计降水 kg m-2 -> 降水率 kg m-2 s-1
        elif target_var == "precip":
            data = data / accumulation_seconds

        hourly_data.append(data)

        if nav_lat is None:
            nav_lat = lat
            nav_lon = lon

        print(
            f"  {valid_time:%Y-%m-%d %H:%M} "
            f"<- P{lead_hour:03d} "
            f"{path.name}"
        )

    all_data[target_var] = np.stack(
        hourly_data,
        axis=0,
    ).astype(np.float32)


# ============================================================
# 建立输出Dataset
# ============================================================

y_size, x_size = nav_lat.shape

ds_out = xr.Dataset(
    coords={
        "time_counter": times,
        "y": np.arange(y_size, dtype=np.int32),
        "x": np.arange(x_size, dtype=np.int32),
    }
)

ds_out["nav_lat"] = (
    ("y", "x"),
    nav_lat,
)

ds_out["nav_lon"] = (
    ("y", "x"),
    nav_lon,
)

ds_out["nav_lat"].attrs = {
    "units": "degrees_north",
    "standard_name": "latitude",
}

ds_out["nav_lon"].attrs = {
    "units": "degrees_east",
    "standard_name": "longitude",
}


for variable_name, data in all_data.items():

    ds_out[variable_name] = (
        ("time_counter", "y", "x"),
        data,
    )

    ds_out[variable_name].attrs["units"] = UNITS[variable_name]


# 云量全部填0
ds_out["percentcloud"] = (
    ("time_counter", "y", "x"),
    np.zeros(
        (len(times), y_size, x_size),
        dtype=np.float32,
    ),
)

ds_out["percentcloud"].attrs["units"] = "percent"


# ============================================================
# 写出NetCDF
# ============================================================

encoding = {
    "time_counter": {
        "units": "seconds since 1970-01-01 00:00:00",
        "calendar": "standard",
        "dtype": "float64",
    }
}

for variable_name in list(VAR_MAPPING) + ["percentcloud"]:
    encoding[variable_name] = {
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,
        "_FillValue": np.float32(1.0e20),
    }


ds_out.to_netcdf(
    OUTPUT_FILE,
    engine="netcdf4",
    unlimited_dims=["time_counter"],
    encoding=encoding,
)

ds_out.close()

print(f"\n生成完毕：{OUTPUT_FILE}")


处理 solar
  2023-03-01 00:00 <- P012 CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20230228T12Z_P012-00.grib2
  2023-03-01 01:00 <- P001 CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20230301T00Z_P001-00.grib2
  2023-03-01 02:00 <- P002 CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20230301T00Z_P002-00.grib2
  2023-03-01 03:00 <- P003 CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20230301T00Z_P003-00.grib2
  2023-03-01 04:00 <- P004 CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20230301T00Z_P004-00.grib2
  2023-03-01 05:00 <- P005 CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20230301T00Z_P005-00.grib2
  2023-03-01 06:00 <- P006 CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20230301T00Z_P006-00.grib2
  2023-03-01 07:00 <- P007 CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20230301T00Z_P007-00.grib2
  2023-03-01 08:00 <- P008 CMC_hrdps_west_DSWRF_SFC_0_rotated_latlon0.009x0.009_20230301T00Z_P008-00.grib2
  2023-03-01 09:00 <- P009 

## Examination



In [12]:
import netCDF4 as nc
import numpy as np

# Define file paths
path_HRDPS_1km = '/home/jqiu/analysis-junqi/Analysis_Atmospheric_Forcing/Data_Conversion/Data_HRDPS_1km_conversion/HRDPS_1km_y2023m03d01.nc'
path_HRDPS = '/results/forcing/atmospheric/continental2.5/nemo_forcing/hrdps_y2023m03d01.nc'

# Target variables for comparison
TARGET_VARS = [
    'time_counter', 'x', 'y', 
    'nav_lat', 'nav_lon', 
    'atmpres', 'percentcloud', 'precip', 
    'qair', 'solar', 'tair', 
    'therm_rad', 'u_wind', 'v_wind'
]

def get_var_info(dataset, var_name):
    """Extract attributes and sample data for a single variable, returning a dictionary."""
    if var_name not in dataset.variables:
        return {"shape": "Missing", "type": "Missing", "units": "Missing", "sample": "Missing"}
    
    var_obj = dataset.variables[var_name]
    info = {
        "shape": str(var_obj.shape),
        "type": str(var_obj.datatype),
        "units": getattr(var_obj, 'units', 'None')  # Display "None" if units attribute is missing
    }
    
    # Safely read a small amount of sample data
    if np.prod(var_obj.shape) > 0:
        try:
            if len(var_obj.shape) == 1:
                sample = var_obj[:3]
            elif len(var_obj.shape) == 2:
                sample = var_obj[0, :3]
            elif len(var_obj.shape) == 3:
                sample = var_obj[0, 0, :3]
            elif len(var_obj.shape) == 4:
                sample = var_obj[0, 0, 0, :3]
            else:
                sample = var_obj[...].flatten()[:3]
            
            # Format array: truncate floats to avoid long strings, take first 3 values
            if sample.dtype.kind == 'f':
                sample_str = "[" + ", ".join([f"{val:.6g}" for val in sample]) + ", ...]"
            else:
                sample_str = "[" + ", ".join([str(val) for val in sample]) + ", ...]"
                
            info["sample"] = sample_str
        except Exception:
            info["sample"] = "Read Error"
    else:
        info["sample"] = "Empty Array"
        
    return info

def compare_nc_files_as_table(file1, file2, vars_to_check):
    print("\nOpening files for side-by-side comparison...")
    try:
        ds1 = nc.Dataset(file1, 'r')
        ds2 = nc.Dataset(file2, 'r')
    except Exception as e:
        print(f"Failed to open files: {e}")
        return
    
    # Set table column widths
    col1_w = 14  # Variable Name
    col2_w = 10  # Attribute
    col3_w = 36  # File 1 Data
    col4_w = 36  # File 2 Data
    
    # Print table header
    sep_line = "-" * (col1_w + col2_w + col3_w + col4_w + 13)
    print(sep_line)
    print(f"| {'Variable':<{col1_w}} | {'Attribute':<{col2_w}} | {'CaSR (File 1)':<{col3_w}} | {'HRDPS (File 2)':<{col4_w}} |")
    print(sep_line)
    
    # Iterate through and compare each variable
    for var in vars_to_check:
        info1 = get_var_info(ds1, var)
        info2 = get_var_info(ds2, var)
        
        # First row: Variable Name and Shape
        print(f"| {var:<{col1_w}} | {'Shape':<{col2_w}} | {info1['shape']:<{col3_w}} | {info2['shape']:<{col4_w}} |")
        # Subsequent rows: Type, Units, Sample (Variable column left blank for clean look)
        print(f"| {'':<{col1_w}} | {'Type':<{col2_w}} | {info1['type']:<{col3_w}} | {info2['type']:<{col4_w}} |")
        print(f"| {'':<{col1_w}} | {'Units':<{col2_w}} | {info1['units']:<{col3_w}} | {info2['units']:<{col4_w}} |")
        print(f"| {'':<{col1_w}} | {'Sample':<{col2_w}} | {info1['sample']:<{col3_w}} | {info2['sample']:<{col4_w}} |")
        print(sep_line)

    ds1.close()
    ds2.close()

# Execute comparison
compare_nc_files_as_table(path_HRDPS_1km, path_HRDPS, TARGET_VARS)


Opening files for side-by-side comparison...
-------------------------------------------------------------------------------------------------------------
| Variable       | Attribute  | CaSR (File 1)                        | HRDPS (File 2)                       |
-------------------------------------------------------------------------------------------------------------
| time_counter   | Shape      | (24,)                                | (24,)                                |
|                | Type       | float64                              | float64                              |
|                | Units      | seconds since 1970-01-01             | seconds since 1970-01-01             |
|                | Sample     | [1.67763e+09, 1.67763e+09, 1.67764e+09, ...] | [1.67763e+09, 1.67763e+09, 1.67764e+09, ...] |
-------------------------------------------------------------------------------------------------------------
| x              | Shape      | (1330,)                   